# Setup

## Load packages

In [1]:
# Load up necessary packages. 
import os
import glob
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Check GPU availability. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Import my custom utils.
import utils
importlib.reload(utils)

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


<module 'utils' from '/tscc/projects/ps-yeolab3/kflanagan/plip_plop/machine_learning_prototype/utils.py'>

## Load results

In [2]:
INPUT_DIR = "/tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/p2_processed_inputs/"

COMPARISON_TABLE = (
    "/tscc/nfs/home/kflanagan/projects/plip_plop/"
    "machine_learning_prototype/encode_initial_30_comparisons.tsv"
)

comparison_table = pd.read_csv(COMPARISON_TABLE, sep="\t")

# Running pytorch

## Data setup

### Create metadata

In [3]:
all_signals = []
all_metadata = []

for comparison_id, row in comparison_table.iterrows():
    experiment_A = row["experiment_A"]
    experiment_B = row["experiment_B"]

    npz_file = os.path.join(INPUT_DIR, f"{experiment_A}_{experiment_B}.npz")

    if not os.path.exists(npz_file):
        print(f"Missing: {npz_file}")
        continue

    data = np.load(npz_file)

    # Load the final abundance-corrected, smoothed signal.
    signals = data["signals"].astype(np.float32)
    n_windows = len(signals)

    all_signals.append(signals)

    metadata = pd.DataFrame({
        "comparison_id": comparison_id,
        "category": row["category"],
        "experiment_A": experiment_A,
        "experiment_B": experiment_B,
        "chrom": data["chrom"],
        "start": data["start"],
        "end": data["end"],
        "strand": data["strand"],
        "region_id": data["region_id"],
        "block_number": data["block_number"],
        "signal_A": data["total_signal_A"],
        "signal_B": data["total_signal_B"],
        "total_signal": data["total_signal"]
    })

    all_metadata.append(metadata)

In [4]:
signals = np.concatenate(all_signals, axis=0)
metadata = pd.concat(all_metadata, ignore_index=True)

# Calculate the maximum left and right shift allowed for each window (trying to keep peaks from falling off the map.)
max_left_shift, max_right_shift = utils.get_shift_bounds(signals)

## Train/val split setup. 

In [5]:
# Create a combined identifier from comparison and region. 
metadata["region_key"] = (
    metadata["comparison_id"].astype(str)
    + "_"
    + metadata["region_id"].astype(str)
)

In [6]:
# Setup random seed. 
rng = np.random.default_rng(42)

# Initialize masking vector. 
train_mask = np.zeros(len(metadata), dtype=bool)
val_mask = np.zeros(len(metadata), dtype=bool)

# Build mask. 
for comparison_id in metadata["comparison_id"].unique():
    comparison_rows = metadata["comparison_id"] == comparison_id

    comparison_regions = (
        metadata.loc[comparison_rows, "region_key"]
        .unique()
        .copy()
    )

    rng.shuffle(comparison_regions)

    split = int(len(comparison_regions) * 0.8)

    train_regions = comparison_regions[:split]
    val_regions = comparison_regions[split:]

    train_mask |= metadata["region_key"].isin(train_regions)
    val_mask |= metadata["region_key"].isin(val_regions)

/scratch/kflanagan/job_12333404/ipykernel_1861124/2334664090.py:18: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(comparison_regions)


In [7]:
# Apply mask to the signals.

train_signals = signals[train_mask]
val_signals = signals[val_mask]

# Apply mask to the shift bounds.

train_max_left_shift = max_left_shift[train_mask]
val_max_left_shift = max_left_shift[val_mask]

train_max_right_shift = max_right_shift[train_mask]
val_max_right_shift = max_right_shift[val_mask]

# Apply mask to the metadata.

train_metadata = metadata.loc[train_mask].reset_index(drop=True)
val_metadata = metadata.loc[val_mask].reset_index(drop=True)

## Setup data loaders. 

In [8]:
# Create special data class for machine learning.
class RBPWindowDataset(Dataset):
    def __init__(self, signals, max_left_shift, max_right_shift):
        self.signals = torch.tensor(signals, dtype=torch.float32)
        self.max_left_shift = max_left_shift
        self.max_right_shift = max_right_shift

    def __len__(self):
        return(len(self.signals))

    def __getitem__(self, idx):
        signal = self.signals[idx]
        left = int(self.max_left_shift[idx])
        right = int(self.max_right_shift[idx])

        view_1 = utils.augment_signal(signal, left, right)
        view_2 = utils.augment_signal(signal, left, right)

        return(view_1, view_2)

train_dataset = RBPWindowDataset(train_signals, train_max_left_shift, train_max_right_shift)
val_dataset = RBPWindowDataset(val_signals, val_max_left_shift, val_max_right_shift)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

In [9]:
view_1, view_2 = next(iter(train_loader))

print("View 1:", view_1.shape)
print("View 2:", view_2.shape)

View 1: torch.Size([1024, 2, 300])
View 2: torch.Size([1024, 2, 300])


## Model setup

In [10]:
# Create VICReg model.
model = utils.VICRegModel(
    latent_dim=64,
    projection_dim=128
).to(device)

# Create optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
view_1, view_2 = next(iter(train_loader))

view_1 = view_1.to(device)
view_2 = view_2.to(device)

with torch.no_grad():
    z_1 = model(view_1)
    z_2 = model(view_2)

print("View 1:", view_1.shape)
print("View 2:", view_2.shape)
print("Projection 1:", z_1.shape)
print("Projection 2:", z_2.shape)

View 1: torch.Size([1024, 2, 300])
View 2: torch.Size([1024, 2, 300])
Projection 1: torch.Size([1024, 128])
Projection 2: torch.Size([1024, 128])


## Run model

In [12]:
train_losses = []
val_losses = []

train_invariance_losses = []
train_variance_losses = []
train_covariance_losses = []

val_invariance_losses = []
val_variance_losses = []
val_covariance_losses = []

best_val_loss = float("inf")
best_model_state = None

num_epochs = 40

for epoch in range(num_epochs):
    model.train()

    total_train_loss = 0
    total_train_invariance = 0
    total_train_variance = 0
    total_train_covariance = 0
    total_train_batches = 0

    for view_1, view_2 in train_loader:
        view_1 = view_1.to(device)
        view_2 = view_2.to(device)

        optimizer.zero_grad()

        z_1 = model(view_1)
        z_2 = model(view_2)

        loss, invariance_loss, variance_loss, covariance_loss = utils.vicreg_loss(
            z_1,
            z_2
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        total_train_invariance += invariance_loss.item()
        total_train_variance += variance_loss.item()
        total_train_covariance += covariance_loss.item()
        total_train_batches += 1

    average_train_loss = total_train_loss / total_train_batches
    average_train_invariance = total_train_invariance / total_train_batches
    average_train_variance = total_train_variance / total_train_batches
    average_train_covariance = total_train_covariance / total_train_batches

    model.eval()

    total_val_loss = 0
    total_val_invariance = 0
    total_val_variance = 0
    total_val_covariance = 0
    total_val_batches = 0

    with torch.no_grad():
        for view_1, view_2 in val_loader:
            view_1 = view_1.to(device)
            view_2 = view_2.to(device)

            z_1 = model(view_1)
            z_2 = model(view_2)

            loss, invariance_loss, variance_loss, covariance_loss = utils.vicreg_loss(
                z_1,
                z_2
            )

            total_val_loss += loss.item()
            total_val_invariance += invariance_loss.item()
            total_val_variance += variance_loss.item()
            total_val_covariance += covariance_loss.item()
            total_val_batches += 1

    average_val_loss = total_val_loss / total_val_batches
    average_val_invariance = total_val_invariance / total_val_batches
    average_val_variance = total_val_variance / total_val_batches
    average_val_covariance = total_val_covariance / total_val_batches

    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)

    train_invariance_losses.append(average_train_invariance)
    train_variance_losses.append(average_train_variance)
    train_covariance_losses.append(average_train_covariance)

    val_invariance_losses.append(average_val_invariance)
    val_variance_losses.append(average_val_variance)
    val_covariance_losses.append(average_val_covariance)

    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1}: "
        f"train = {average_train_loss:.6f}, "
        f"validation = {average_val_loss:.6f} | "
        f"train inv = {average_train_invariance:.6f}, "
        f"var = {average_train_variance:.6f}, "
        f"cov = {average_train_covariance:.6f}"
    )

Epoch 1: train = 15.967705, validation = 16.373836 | train inv = 0.053828, var = 0.484757, cov = 2.503077
Epoch 2: train = 13.087338, validation = 15.736056 | train inv = 0.048074, var = 0.337891, cov = 3.438203
Epoch 3: train = 11.636563, validation = 15.310679 | train inv = 0.051204, var = 0.258084, cov = 3.904366
Epoch 4: train = 11.003933, validation = 14.595467 | train inv = 0.048948, var = 0.225180, cov = 4.150732
Epoch 5: train = 10.549914, validation = 14.965834 | train inv = 0.049287, var = 0.201710, cov = 4.274991
Epoch 6: train = 10.142628, validation = 14.324425 | train inv = 0.049744, var = 0.179659, cov = 4.407558
Epoch 7: train = 9.728046, validation = 14.793132 | train inv = 0.049062, var = 0.158366, cov = 4.542351
Epoch 8: train = 9.458386, validation = 14.350132 | train inv = 0.049197, var = 0.142892, cov = 4.656166
Epoch 9: train = 9.286183, validation = 14.143485 | train inv = 0.048346, var = 0.134052, cov = 4.726230
Epoch 10: train = 9.075234, validation = 14.35546

In [13]:
# Loads best model, not just last model. 
model.load_state_dict(best_model_state)
model = model.to(device)

# Save model for later use. 
torch.save(model.state_dict(), "/tscc/nfs/home/kflanagan/scratch/plip_plop_results/VICreg_test.pt")

# Basic testing

In [14]:
# Set model to evaluation mode.
model.eval()

# -------------------------------------------------------------------------
# 1. Check for representation collapse.
# -------------------------------------------------------------------------

val_tensor = torch.tensor(val_signals, dtype=torch.float32)
latent_batches = []

with torch.no_grad():
    for start in range(0, len(val_tensor), 1024):
        batch = val_tensor[start:start + 1024].to(device)
        latent = model.encode(batch)
        latent_batches.append(latent.cpu())

val_latents = torch.cat(latent_batches, dim=0)

# Calculate the standard deviation of every latent dimension.
latent_std = val_latents.std(dim=0)

print("Latent shape:", val_latents.shape)
print("Mean latent SD:", latent_std.mean().item())
print("Minimum latent SD:", latent_std.min().item())
print("Maximum latent SD:", latent_std.max().item())
print("Dimensions with SD < 0.01:", (latent_std < 0.01).sum().item())
print("Dimensions with SD < 0.1:", (latent_std < 0.1).sum().item())

# Check how redundant the latent dimensions are.
centered_latents = val_latents - val_latents.mean(dim=0)
latent_cov = torch.cov(centered_latents.T)

latent_corr = torch.corrcoef(centered_latents.T)
off_diagonal_corr = utils.off_diagonal(latent_corr)

print("Mean absolute off-diagonal correlation:", off_diagonal_corr.abs().mean().item())
print("Maximum absolute off-diagonal correlation:", off_diagonal_corr.abs().max().item())

# Estimate the effective dimensionality of the representation.
eigenvalues = torch.linalg.eigvalsh(latent_cov)
eigenvalues = torch.clamp(eigenvalues, min=0)

normalized_eigenvalues = eigenvalues / eigenvalues.sum()
effective_rank = torch.exp(
    -(normalized_eigenvalues * torch.log(normalized_eigenvalues + 1e-12)).sum()
)

print("Effective rank:", effective_rank.item())


# -------------------------------------------------------------------------
# 2. Check whether augmented views of the same window remain similar.
# -------------------------------------------------------------------------

n_test_windows = min(1000, len(val_signals))

same_window_distances = []
different_window_distances = []

with torch.no_grad():
    for i in range(n_test_windows):
        signal = torch.tensor(val_signals[i], dtype=torch.float32)

        # Create two independently augmented views of the same window.
        view_1 = utils.augment_signal(signal).unsqueeze(0).to(device)
        view_2 = utils.augment_signal(signal).unsqueeze(0).to(device)

        latent_1 = model.encode(view_1)
        latent_2 = model.encode(view_2)

        same_distance = torch.norm(latent_1 - latent_2, p=2).item()
        same_window_distances.append(same_distance)

        # Compare against a different validation window.
        j = (i + 1) % len(val_signals)
        different_signal = torch.tensor(val_signals[j], dtype=torch.float32)
        different_signal = different_signal.unsqueeze(0).to(device)

        different_latent = model.encode(different_signal)

        different_distance = torch.norm(latent_1 - different_latent, p=2).item()
        different_window_distances.append(different_distance)

same_window_distances = np.array(same_window_distances)
different_window_distances = np.array(different_window_distances)

print()
print("Mean distance between augmented views of same window:",
      same_window_distances.mean())

print("Mean distance between different windows:",
      different_window_distances.mean())

print("Median distance between augmented views of same window:",
      np.median(same_window_distances))

print("Median distance between different windows:",
      np.median(different_window_distances))

print(
    "Fraction where same-window views are closer than different windows:",
    np.mean(same_window_distances < different_window_distances)
)

Latent shape: torch.Size([143119, 64])
Mean latent SD: 0.689346194267273
Minimum latent SD: 0.24942053854465485
Maximum latent SD: 1.6684542894363403
Dimensions with SD < 0.01: 0
Dimensions with SD < 0.1: 0
Mean absolute off-diagonal correlation: 0.2746537923812866
Maximum absolute off-diagonal correlation: 0.9531697034835815
Effective rank: 7.19508171081543


TypeError: augment_signal() missing 2 required positional arguments: 'max_left_shift' and 'max_right_shift'